# LLM Feature Extraction — Calibration Run (30 articles)

One-shot validation notebook. Runs the new extraction prompt on a stratified
30-article sample and saves results to `05_reports/calibration/llm_calibration.json`.

**Do not re-run after the calibration is approved** — output is a fixed validation
artifact, not a recurring pipeline step. Production extraction is in `06_llm_features.ipynb`.

Strata in `calibration_sample`:
- `body_valid=1` — 10 articles (passed regex filter)
- `body_valid=0` substantial — 10 articles (rejected by regex but human-judged to have real content)
- `body_valid=0` weak — 10 articles (rejected by regex, minimal content)

### Imports and setup

In [ ]:
import sys
import os
import json
import sqlite3
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
import anthropic

load_dotenv(Path('../.env').resolve())

sys.path.insert(0, str(Path('../03_src').resolve()))
from nlp.llm_features import extract_features

client = anthropic.Anthropic(api_key=os.environ['ANTHROPIC_API_KEY'])
print('Client ready')

### Load calibration sample from DB

In [ ]:
conn = sqlite3.connect('../01_data/wti_thesis.db')
df = pd.read_sql(
    'SELECT id, title, body, body_valid, bucket FROM calibration_sample',
    conn
)
conn.close()

print(f'Loaded {len(df)} articles')
print(df.groupby('bucket').size().to_string())

### Run extraction

Synchronous, no batching — 30 articles. Results are NOT written to the DB.
Output goes to `05_reports/calibration/llm_calibration.json` only.

In [ ]:
results = {}
errors = {}

for _, row in df.iterrows():
    article_id = int(row['id'])
    try:
        features = extract_features(row['title'], row['body'], client)
        results[article_id] = {
            'id': article_id,
            'bucket': row['bucket'],
            'body_valid': int(row['body_valid']) if row['body_valid'] is not None else None,
            'title': row['title'],
            **features,
        }
        usable_str = '✓' if features['usable'] else '✗'
        print(f'[{len(results):>2}/{len(df)}] {usable_str}  {row["title"][:70]}')
    except Exception as e:
        errors[article_id] = str(e)
        print(f'ERROR id={article_id}: {e}')

print(f'\nDone — {len(results)} extracted, {len(errors)} errors')

### Save to JSON

In [ ]:
out_path = Path('../05_reports/calibration/llm_calibration.json')
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, 'w') as f:
    json.dump(results, f, indent=2, default=str)

print(f'Saved: {out_path}  ({out_path.stat().st_size:,} bytes)')

### Summary statistics

In [ ]:
df_res = pd.DataFrame(results.values())

n_usable = df_res['usable'].sum()
n_not_usable = (~df_res['usable']).sum()
print(f'usable=true:  {n_usable}')
print(f'usable=false: {n_not_usable}')

print('\n--- usable by bucket ---')
print(df_res.groupby('bucket')['usable'].value_counts().to_string())

print('\n--- usable by body_valid ---')
print(df_res.groupby('body_valid')['usable'].value_counts().to_string())

usable_rows = df_res[df_res['usable']]
if len(usable_rows) > 0:
    print('\n--- event_type distribution (usable articles) ---')
    from collections import Counter
    all_types = [t for row in usable_rows['event_type'].dropna() for t in row]
    for k, v in sorted(Counter(all_types).items(), key=lambda x: -x[1]):
        print(f'  {k}: {v}')

    print('\n--- sentiment_score stats (usable articles) ---')
    print(usable_rows['sentiment_score'].describe().round(3).to_string())

if errors:
    print(f'\n--- extraction errors ({len(errors)}) ---')
    for aid, msg in errors.items():
        print(f'  id={aid}: {msg}')